In [ ]:
import spacy
from fastcoref import spacy_component
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']}")

In [ ]:
import spacy
from fastcoref import spacy_component
import requests

for podcast in podcasts:
    if not podcast["language"].startswith("en"):
        #print(f"skipping {podcast['title']}, language is {podcast['language']}")
        continue
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()
                    utterances = seg["utterance_set"]
                    # check if coref already exists in any of this segmentation's utterances
                    if any([utt["text_coref"] for utt in utterances]):
                        #print("skipping ", audioitem['title'], podcast['title'])
                        continue
                    
                    print("starting episode: ", audioitem['title'], podcast['title'])
                    nlp = spacy.load("en_core_web_lg")
                    nlp.add_pipe(
                        "fastcoref", 
                        config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
                    )

                    # add a context field to each utterance with the 50 previous utterances
                    for i, utt in enumerate(utterances):
                        utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-30):i+1]])

                    docs = nlp.pipe(
                    [utterance["context"] for utterance in utterances],
                    component_cfg={"fastcoref": {'resolve_text': True}}
                    )
                    #docs = list(docs)
                    try:
                        docs = list(docs)
                    except:
                        print("error with ", audioitem['title'], podcast['title'])
                        for doc in docs:
                            print(doc)
                            print(utterances)
                        continue

                    assert len(docs) == len(utterances)

                    for i, doc in enumerate(docs):
                        resolved_text = doc._.resolved_text

                        # Create a new Doc object without running the entire pipeline
                        sentences_doc = nlp.make_doc(resolved_text)

                        # Apply the "senter" component to the sentences_doc
                        nlp.get_pipe("senter")(sentences_doc)

                        first = coref = next(sentences_doc.sents)
                        for coref in sentences_doc.sents: pass

                        first = original = next(doc.sents)
                        for original in doc.sents: pass
                        if coref.text.strip() != original.text.strip():
                            # write to API
                            utt_uuid = utterances[i]["uuid"]
                            utterances[i]["text_coref"] = coref.text.strip()
                            # no changes to these child record, so remove them
                            utterances[i].pop("classification_set")
                            utterances[i].pop("query_set")
                            res = requests.post(f"{SERVER}:{PORT}/api/utterances/{utt_uuid}/", json=utterances[i])
                            print(res.status_code)

                    

In [ ]:
nlp = spacy.load("en_core_web_lg", exclude=["parser", "lemmatizer", "ner", "textcat"])
nlp.add_pipe(
    "fastcoref", 
    config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
)
docs = nlp.pipe(
[utterance["context"] for utterance in utterances],
component_cfg={"fastcoref": {'resolve_text': True}}
)

In [ ]:
for doc in docs:
    try:
        print(doc)
    except:
        print("!!!!!!!!!!!!!!!!!!!!!error")
        print(doc)

In [ ]:
utterances[i].pop("classification_set")

In [ ]:
utterances[i]

In [ ]:
[utterance["context"] for utterance in utterances]

In [ ]:
import copy
for podcast in podcasts:
    if not podcast["language"].startswith("en"):
        continue
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()
                    utterances = seg["utterance_set"]
                    if any([utt["text_coref"] for utt in utterances]):
                        continue
                    
                    print("starting episode: ", audioitem['title'], podcast['title'])
                    nlp = spacy.load("en_core_web_lg")
                    nlp.add_pipe(
                        "fastcoref", 
                        config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
                    )

                    for i, utt in enumerate(utterances):
                        utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-40):i+1]])

                    for utterance in utterances:
                        try:
                            doc = nlp(utterance["context"], component_cfg={"fastcoref": {'resolve_text': True}})
                        except Exception as e:
                            print(f"Error occurred during processing podcast {podcast['title']} with episode {audioitem['title']} and utterance {utterance['context']}:")
                            print(e)
                            continue

                        resolved_text = doc._.resolved_text
                        sentences_doc = nlp.make_doc(resolved_text)
                        nlp.get_pipe("senter")(sentences_doc)

                        first = coref = next(sentences_doc.sents)
                        for coref in sentences_doc.sents: pass

                        first = original = next(doc.sents)
                        for original in doc.sents: pass

                        if coref.text.strip() != original.text.strip():
                            utt_uuid = utterance["uuid"]

                            # Deep copy the utterance
                            utterance_copy = copy.deepcopy(utterance)

                            utterance_copy["text_coref"] = coref.text.strip()
                            if utterance_copy.get("classification_set"):
                                utterance_copy.pop("classification_set")
                            if utterance_copy.get("query_set"):
                                utterance_copy.pop("query_set")
                                                    
                            print(utterance_copy)
                            res = requests.post(f"{SERVER}:{PORT}/api/utterances/{utt_uuid}/", json=utterance_copy)
                            print(res.status_code)


In [ ]:
import spacy
from fastcoref import spacy_component
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']}")

In [ ]:
podcast = podcasts[0]
audioitem = [a for a in podcast['audioitem_set'] if a['guid'] == "01c17681-f1d1-4b4e-bac8-b000000ec0f8"][0]
my_seg = "0ac504bc-f0bc-11ed-a5e2-00155d08852a"
seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{my_seg}/")
seg = seg.json()
utterances = seg["utterance_set"]


print("starting episode: ", audioitem['title'], podcast['title'])
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe(
    "fastcoref", 
    config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
)


In [ ]:

for i, utt in enumerate(utterances):
    utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-40):i+1]])

for utterance in utterances:
    try:
        doc = nlp(utterance["context"], component_cfg={"fastcoref": {'resolve_text': True}})
    except Exception as e:
        print(f"Error occurred during processing podcast {podcast['title']} with episode {audioitem['title']} and utterance {utterance['context']}:")
        print(e)
        continue

    resolved_text = doc._.resolved_text
    sentences_doc = nlp.make_doc(resolved_text)
    nlp.get_pipe("senter")(sentences_doc)

    first = coref = next(sentences_doc.sents)
    for coref in sentences_doc.sents: pass

    first = original = next(doc.sents)
    for original in doc.sents: pass